In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import random
import numpy as np
import torch

seed = 53

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [14]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset
import pandas as pd


class AADBIASDataset(Dataset):
    def __init__(
        self,
        csv_path,
        img_root,
        transform=None,
        log_missing_path=None,  # 如果想把 missing id 存成檔案就給路徑
    ):
        """
        csv_path: 'aadb_ias_cLFPO_labels.csv'
        img_root: folder where ALL AADB images are stored
        """
        self.df = pd.read_csv(csv_path)
        self.img_root = img_root
        self.transform = transform

        # ---- 1) 先掃描資料夾，建立 image_id -> filename 對應 ----
        all_files = [
            f for f in os.listdir(img_root)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

        self.id_to_file = {}
        for fname in all_files:
            stem = os.path.splitext(fname)[0]  # 去掉副檔名
            parts = stem.split("_")
            # 依你現在的檔名型態，假設最後三段是 image_id: photoid_hash_b
            if len(parts) >= 3:
                image_id = "_".join(parts[-3:])
                self.id_to_file[image_id] = fname

        # ---- 2) 把 csv 裡找不到圖片的 id 標記起來並丟掉 ----
        all_valid_ids = set(self.id_to_file.keys())
        mask = self.df["image_id"].isin(all_valid_ids)

        self.missing_ids = self.df.loc[~mask, "image_id"].tolist()
        num_missing = len(self.missing_ids)

        # 只保留有對應圖片的 row
        self.df = self.df.loc[mask].reset_index(drop=True)

        print(f"[AADBIASDataset] Kept {len(self.df)} samples, "
              f"dropped {num_missing} samples without image files.")

        # 如果要把 missing id 存成檔案
        if log_missing_path is not None and num_missing > 0:
            with open(log_missing_path, "w") as f:
                for mid in self.missing_ids:
                    f.write(str(mid) + "\n")
            print(f"[AADBIASDataset] Missing image ids saved to {log_missing_path}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row["image_id"]

        fname = self.id_to_file[image_id]
        img_path = os.path.join(self.img_root, fname)

        img = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)

        labels = torch.tensor([
            row["IAS"],
            row["C"],
            row["L"],
            row["F"],
            row["P"],
            row["O"],
        ], dtype=torch.float32)

        return img, labels


In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models


class AestheticMultiHeadNet(nn.Module):
    """
    Input : image tensor of shape (B, 3, H, W)
    Output: dict with keys:
        - 'IAS': (B, 1)  Image Aesthetic Score, range [0, 10]
        - 'C'  : (B, 1)  Composition,        range [0, 10]
        - 'L'  : (B, 1)  Lighting / Color,    range [0, 10]
        - 'F'  : (B, 1)  Focus / Clarity,     range [0, 10]
        - 'P'  : (B, 1)  Post-processing,     range [0, 10]
        - 'O'  : (B, 1)  Originality / Story, range [0, 10]
    """

    def __init__(self, backbone_name: str = "resnet18", pretrained: bool = True):
        super().__init__()

        # ---- 1. Backbone (ResNet) ----
        if backbone_name == "resnet18":
            backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
            feat_dim = 512
        elif backbone_name == "resnet34":
            backbone = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)
            feat_dim = 512
        elif backbone_name == "resnet50":
            backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
            feat_dim = 2048
        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")

        # 移除原本的分類 head，只留下 feature extractor
        modules = list(backbone.children())[:-1]  # 去掉最後的 fc
        self.feature_extractor = nn.Sequential(*modules)

        # ---- 2. Shared embedding layer ----
        hidden_dim = 256
        self.shared_head = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
        )

        # ---- 3. Output heads ----
        # 3-1. Image Aesthetic Score (單一 scalar)
        self.score_head = nn.Linear(hidden_dim, 1)

        # 3-2. Five attributes: C, L, F, P, O
        self.attr_head = nn.Linear(hidden_dim, 5)

    def forward(self, x):
        # x: (B, 3, H, W)

        # ResNet feature: (B, feat_dim, 1, 1)
        feat = self.feature_extractor(x)
        feat = feat.view(feat.size(0), -1)  # (B, feat_dim)

        # Shared embedding
        h = self.shared_head(feat)  # (B, hidden_dim)

        # Outputs
        ias = self.score_head(h)         # (B, 1)
        attrs = self.attr_head(h)        # (B, 5)

        # 把輸出壓到 0~10：sigmoid ∈ (0,1) 再 *10
        ias = torch.sigmoid(ias) * 10.0
        attrs = torch.sigmoid(attrs) * 10.0  # 每個維度都是 0~10

        # 拆成五個名字
        C = attrs[:, 0:1]
        L = attrs[:, 1:2]
        F = attrs[:, 2:3]
        P = attrs[:, 3:4]
        O = attrs[:, 4:5]

        return {
            "IAS": ias,
            "C": C,
            "L": L,
            "F": F,
            "P": P,
            "O": O,
        }


if __name__ == "__main__":
    # 小測試
    model = AestheticMultiHeadNet(backbone_name="resnet18", pretrained=False)
    dummy = torch.randn(4, 3, 224, 224)
    out = model(dummy)
    for k, v in out.items():
        print(k, v.shape, v.min().item(), v.max().item())


IAS torch.Size([4, 1]) 4.320771217346191 5.351598262786865
C torch.Size([4, 1]) 5.040624141693115 5.965246677398682
L torch.Size([4, 1]) 5.654112815856934 6.268094062805176
F torch.Size([4, 1]) 4.0554375648498535 5.230257511138916
P torch.Size([4, 1]) 4.403078556060791 4.798050403594971
O torch.Size([4, 1]) 3.667978048324585 4.5559468269348145


In [16]:
from torchvision import transforms
import pandas as pd
import os

csv_full = "/content/drive/MyDrive/MLFinal/aadb_image_labels.csv"
img_root = "/content/drive/MyDrive/MLFinal/AADB_newtest"

# 定義 train / val 用的 transform
train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1,
        hue=0.02,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# 讀原始 CSV，打亂後切 80% / 20%
df_full = pd.read_csv(csv_full)
df_full = df_full.sample(frac=1.0, random_state=seed).reset_index(drop=True)

n_total = len(df_full)
n_val = int(0.2 * n_total)
df_val = df_full.iloc[:n_val].reset_index(drop=True)
df_train = df_full.iloc[n_val:].reset_index(drop=True)

train_csv = "/content/drive/MyDrive/MLFinal/aadb_train.csv"
val_csv   = "/content/drive/MyDrive/MLFinal/aadb_val.csv"

df_train.to_csv(train_csv, index=False)
df_val.to_csv(val_csv, index=False)

print("Total:", n_total,
      "train:", len(df_train),
      "val:", len(df_val))


Total: 9958 train: 7967 val: 1991


In [17]:
from torch.utils.data import DataLoader

train_set = AADBIASDataset(
    csv_path=train_csv,
    img_root=img_root,
    transform=train_tf,
    log_missing_path="/content/drive/MyDrive/MLFinal/missing_train_ids.txt",
)

val_set = AADBIASDataset(
    csv_path=val_csv,
    img_root=img_root,
    transform=val_tf,
    log_missing_path="/content/drive/MyDrive/MLFinal/missing_val_ids.txt",
)

train_loader = DataLoader(
    train_set,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    val_set,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)


[AADBIASDataset] Kept 789 samples, dropped 7178 samples without image files.
[AADBIASDataset] Missing image ids saved to /content/drive/MyDrive/MLFinal/missing_train_ids.txt
[AADBIASDataset] Kept 211 samples, dropped 1780 samples without image files.
[AADBIASDataset] Missing image ids saved to /content/drive/MyDrive/MLFinal/missing_val_ids.txt


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [18]:
import torch.nn.functional as F
import torch.nn as nn

model = AestheticMultiHeadNet(backbone_name="resnet18", pretrained=True)
model = model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

num_epochs = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs
)

save_path = "/content/drive/MyDrive/MLFinal/aadb_resnet18_multitask_best.pt"



In [8]:
from torch.cuda.amp import GradScaler, autocast
scaler = GradScaler()

best_val_mae = float("inf")
names = ["IAS", "C", "L", "F", "P", "O"]

for epoch in range(1, num_epochs + 1):
    # ---- Training ----
    model.train()
    running_loss = 0.0
    n_train_samples = 0

    for imgs, labels in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with autocast():
            out = model(imgs)

            pred = torch.cat(
                [out["IAS"], out["C"], out["L"], out["F"], out["P"], out["O"]],
                dim=1
            )

            # regression loss：IAS 比較重要
            pred_ias = pred[:, 0]
            true_ias = labels[:, 0]

            loss_ias = F.smooth_l1_loss(pred_ias, true_ias)
            loss_attr = F.mse_loss(pred[:, 1:], labels[:, 1:])

            loss = 1.5 * loss_ias + 1.0 * loss_attr

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = imgs.size(0)
        running_loss += loss.item() * batch_size
        n_train_samples += batch_size

    avg_train_loss = running_loss / n_train_samples

    # ---- Validation ----
    model.eval()
    mae_sum = torch.zeros(6, device=device)
    n_val_samples = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            out = model(imgs)
            pred = torch.cat(
                [out["IAS"], out["C"], out["L"], out["F"], out["P"], out["O"]],
                dim=1
            )

            diff = pred - labels
            mae_sum += diff.abs().sum(dim=0)
            n_val_samples += labels.size(0)

    val_mae_per_attr = (mae_sum / n_val_samples).detach().cpu()
    val_mae = val_mae_per_attr.mean().item()

    # scheduler
    scheduler.step()

    print(f"\nEpoch [{epoch}/{num_epochs}]")
    print(f"  Train loss: {avg_train_loss:.4f}")
    for i, name in enumerate(names):
        print(f"  Val {name} MAE: {val_mae_per_attr[i]:.3f}")
    print(f"  Val MAE (avg 6 dims): {val_mae:.3f}")

    # 儲存best model
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        torch.save(model.state_dict(), save_path)
        print(f"  >> New best model saved (val MAE = {best_val_mae:.3f})")

print("\nTraining finished. Best val MAE:", best_val_mae)
print("Best model path:", save_path)


/tmp/ipython-input-3765202218.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/tmp/ipython-input-3765202218.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # 若不想用混合精度就把 with autocast(): 拿掉
/usr/local/lib/python3.12/dist-packages/torch/amp/autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(



Epoch [1/20]
  Train loss: 11.6126
  Val IAS MAE: 2.011
  Val C MAE: 2.597
  Val L MAE: 2.510
  Val F MAE: 2.694
  Val P MAE: 2.998
  Val O MAE: 2.636
  Val MAE (avg 6 dims): 2.574
  >> New best model saved (val MAE = 2.574)

Epoch [2/20]
  Train loss: 8.3217
  Val IAS MAE: 1.400
  Val C MAE: 2.301
  Val L MAE: 2.159
  Val F MAE: 2.580
  Val P MAE: 2.532
  Val O MAE: 2.416
  Val MAE (avg 6 dims): 2.231
  >> New best model saved (val MAE = 2.231)

Epoch [3/20]
  Train loss: 6.7370
  Val IAS MAE: 1.441
  Val C MAE: 2.399
  Val L MAE: 2.235
  Val F MAE: 2.589
  Val P MAE: 2.538
  Val O MAE: 2.452
  Val MAE (avg 6 dims): 2.276

Epoch [4/20]
  Train loss: 5.7570
  Val IAS MAE: 1.407
  Val C MAE: 2.389
  Val L MAE: 2.200
  Val F MAE: 2.540
  Val P MAE: 2.494
  Val O MAE: 2.495
  Val MAE (avg 6 dims): 2.254

Epoch [5/20]
  Train loss: 4.6889
  Val IAS MAE: 1.366
  Val C MAE: 2.375
  Val L MAE: 2.195
  Val F MAE: 2.558
  Val P MAE: 2.596
  Val O MAE: 2.383
  Val MAE (avg 6 dims): 2.246

Epoch

In [9]:
model.load_state_dict(torch.load(save_path, map_location=device))
model.to(device)
model.eval()

mae_sum = torch.zeros(6, device=device)
mse_sum = torch.zeros(6, device=device)
n_samples = 0

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)  # (B, 6)

        out = model(imgs)
        pred = torch.cat(
            [out["IAS"], out["C"], out["L"], out["F"], out["P"], out["O"]],
            dim=1
        )

        diff = pred - labels
        mae_sum += diff.abs().sum(dim=0)
        mse_sum += (diff ** 2).sum(dim=0)
        n_samples += labels.size(0)

mae = (mae_sum / n_samples).cpu()
rmse = torch.sqrt(mse_sum / n_samples).cpu()

names = ["IAS", "C", "L", "F", "P", "O"]
for i, name in enumerate(names):
    print(f"{name}: MAE = {mae[i]:.3f}, RMSE = {rmse[i]:.3f}")

print("Overall MAE :", mae.mean().item())
print("Overall RMSE:", rmse.mean().item())


IAS: MAE = 1.338, RMSE = 1.653
C: MAE = 2.317, RMSE = 2.863
L: MAE = 2.126, RMSE = 2.725
F: MAE = 2.589, RMSE = 3.262
P: MAE = 2.454, RMSE = 3.093
O: MAE = 2.466, RMSE = 3.098
Overall MAE : 2.2150323390960693
Overall RMSE: 2.7822189331054688


In [ ]:
import os
from PIL import Image
import pandas as pd
import numpy as np

val_root = "/content/drive/MyDrive/MLFinal/validation"
pred_csv_path = "/content/drive/MyDrive/MLFinal/validation_predictions.csv"

model.eval()

def predict_image(img_path):
    img = Image.open(img_path).convert("RGB")
    x = val_tf(img).unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(x)

    scores = {k: v.item() for k, v in out.items()}
    return scores

results = []

for set_name in sorted(os.listdir(val_root)):
    set_dir = os.path.join(val_root, set_name)
    if not os.path.isdir(set_dir):
        continue

    for fname in sorted(os.listdir(set_dir)):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        img_path = os.path.join(set_dir, fname)
        scores = predict_image(img_path)

        row = {
            "set": set_name,
            "filename": fname,
            "IAS": scores["IAS"],
            "C": scores["C"],
            "L": scores["L"],
            "F": scores["F"],
            "P": scores["P"],
            "O": scores["O"],
        }
        results.append(row)

        print(f"{set_name}/{fname} -> "
              f"IAS={scores['IAS']:.2f}, "
              f"C={scores['C']:.2f}, "
              f"L={scores['L']:.2f}, "
              f"F={scores['F']:.2f}, "
              f"P={scores['P']:.2f}, "
              f"O={scores['O']:.2f}")

df_pred = pd.DataFrame(results)
df_pred.to_csv(pred_csv_path, index=False)
print("Saved predictions to:", pred_csv_path)


In [11]:
df = pd.read_csv(pred_csv_path)
df["type"] = np.where(df["filename"].str.contains("good"), "good", "bad")

print(df.groupby(["set", "type"]).size())

records = []
for s, group in df.groupby("set"):
    good_row = group[group["type"] == "good"].iloc[0]
    bad_row  = group[group["type"] == "bad"].iloc[0]

    good_ias = good_row["IAS"]
    bad_ias  = bad_row["IAS"]
    margin   = good_ias - bad_ias
    correct  = margin > 0

    records.append({
        "set": s,
        "good_IAS": good_ias,
        "bad_IAS": bad_ias,
        "margin": margin,
        "correct": int(correct),
    })

eval_df = pd.DataFrame(records)
print(eval_df)

pairwise_acc = eval_df["correct"].mean()
mean_margin  = eval_df["margin"].mean()

print(f"\nPairwise accuracy (IAS_good > IAS_bad): {pairwise_acc*100:.1f}%")
print(f"Average IAS margin (good - bad): {mean_margin:.3f}")


set    type
set1   bad     1
       good    1
set10  bad     1
       good    1
set2   bad     1
       good    1
set3   bad     1
       good    1
set4   bad     1
       good    1
set5   bad     1
       good    1
set6   bad     1
       good    1
set7   bad     1
       good    1
set8   bad     1
       good    1
dtype: int64
     set  good_IAS   bad_IAS    margin  correct
0   set1  7.086740  2.787169  4.299571        1
1  set10  5.891729  6.714184 -0.822455        0
2   set2  6.715025  6.275054  0.439971        1
3   set3  6.886931  3.570341  3.316590        1
4   set4  5.969162  4.629200  1.339962        1
5   set5  7.011788  4.890609  2.121179        1
6   set6  6.451916  5.184603  1.267313        1
7   set7  6.120734  4.856381  1.264353        1
8   set8  7.899575  4.920441  2.979135        1

Pairwise accuracy (IAS_good > IAS_bad): 88.9%
Average IAS margin (good - bad): 1.801
